In [ ]:
""" Model Monitoring using Evidently

Model monitoring tracks a machine learning model's performance, predictions, and input data to detect issues such as drift in real time.

It ensures:
1. Model reliability
2. Identifies data drift
3. Enables timely maintenance

Objectives:
- Log inference predictions from the model.
- Generate Evidently AI reports to monitor model performance and data drift.

Installation: pip install evidently
"""

import pandas as pd
import torch
from time import time
import os
import torch.nn as nn
import pickle
import re
import evidently
from evidently import Report
from evidently.presets.classification import ClassificationPreset
from evidently.presets.drift import DataDriftPreset
import pandas as pd
from evidently import Dataset
from evidently import DataDefinition
from evidently import MulticlassClassification

In [ ]:
ROOT_DIR = r'E:\PyCharmProjects\BongoDev ML Bootcamp\MyABSAService'

In [ ]:
dictionary_path = ROOT_DIR + r'\vocab.pkl'
model_path = ROOT_DIR + r'\model_weights.pth'

In [ ]:
# Load dictionary
token_2_id = None
with open(ROOT_DIR + r'\vocab.pkl', "rb") as f:
    token_2_id = pickle.load(f)

In [ ]:
# Normalize
def normalize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = ' '.join(text.split())
    return text

In [ ]:
# Tokenize
def tokenize(text):
    tokens = text.split()
    return tokens

In [ ]:
# Convert tokens into ids
def convert_tokens_2_ids(tokens):
    input_ids = [
        token_2_id.get(token, token_2_id['<UNK>']) for token in tokens
    ]
    return input_ids

In [ ]:
# Process an input text
def process_text(text, aspect):
    text_aspect_pair = text + ' ' + aspect
    normalized_text = normalize(text_aspect_pair)
    tokens = tokenize(normalized_text)
    input_ids = convert_tokens_2_ids(tokens)
    input_ids = torch.tensor(input_ids).unsqueeze(0)
    return input_ids

In [ ]:
# ABSA Model
class ABSA(nn.Module):
    def __init__(self, vocab_size, num_labels=3):
        super(ABSA, self).__init__()
        self.vocab_size = vocab_size
        self.num_labels = num_labels
        self.embedding_layer = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=256
        )
        self.lstm_layer = nn.LSTM(
            input_size=256,
            hidden_size=512,
            batch_first=True,
        )

        self.fc_layer = nn.Linear(
            in_features=512,
            out_features=self.num_labels
        )

    def forward(self, x):
        embeddings = self.embedding_layer(x)
        lstm_out, _ = self.lstm_layer(embeddings)
        logits = self.fc_layer(lstm_out[:, -1, :])
        return logits

model = ABSA(
    vocab_size=len(token_2_id.keys()),
    num_labels=3
)
model.load_state_dict(
    torch.load(model_path)
)
model.eval()
print("Model loaded successfully")

Model loaded successfully


In [ ]:
SENTIMENT_LABELS = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

def log_prediction(text, aspect, sentiment, label, confidence, inference_time):
    metrics = pd.read_csv(
        ROOT_DIR + r'\inference_metrics.csv'
    )
    new_row = {
        'timestamp': pd.Timestamp.now(),
        'text': text,
        'aspect': aspect,
        'sentiment': sentiment, # Negative, Neutral, Positive
        'target': "",
        'prediction': label, # 0, 1, 2
        'confidence': confidence, # probability [0.15, 0.65, 0.20]
        'inference_time': inference_time,
    }
    metrics = pd.concat(
        [metrics, pd.DataFrame([new_row])],
        ignore_index=True
    )

    try:
        metrics.to_csv(ROOT_DIR + r'\inference_metrics.csv', index=False)
    except Exception as e:
        print(e)

def predict_sentiment(text, aspect):
    start_time = time()
    input_ids = process_text(text, aspect)
    with torch.no_grad():
        logits = model(input_ids)
        inference_time = time() - start_time
        probs = torch.softmax(logits, dim=-1)
        label = probs.argmax(dim=-1).item()
        sentiment = SENTIMENT_LABELS[label],
        confidence = probs.squeeze().tolist()[label]
        log_prediction(
            text,
            aspect,
            sentiment,
            label,
            confidence,
            inference_time
        )
        return {"sentiment": sentiment, "confidence": confidence}

text = "The food was great but the service was terrible. The environment was very good."
aspects = ["food", "service", "ambience"]
for aspect in aspects:
    result = predict_sentiment(text, aspect)
    print(f"{aspect}: {result['sentiment']} (Confidence: {result['confidence']:.4f})")

food: ('Positive',) (Confidence: 0.7412)
service: ('Positive',) (Confidence: 0.7815)
ambience: ('Positive',) (Confidence: 0.8477)


# Model Monitoring

In [ ]:
reference_df = pd.read_csv(ROOT_DIR + r'\reference_data.csv')
current_df = pd.read_csv(ROOT_DIR + r'\inference_metrics.csv')

In [ ]:
current_df['prediction'].value_counts()

prediction
2    4
1    1
0    1
Name: count, dtype: int64

In [ ]:
columns = ['target', 'prediction']
reference_df = reference_df[columns]
current_df = current_df[columns]

In [ ]:
# Prepare data for monitoring
data_def = DataDefinition(
    classification=[
        MulticlassClassification(
            target="target",
            prediction_labels="prediction"
        )
    ]
)



reference_data = Dataset.from_pandas(
    reference_df,
    data_definition=data_def,
)
current_data = Dataset.from_pandas(
    current_df,
    data_definition=data_def,
)


In [ ]:
# Create Performance Report

classification_report = Report(
    metrics=[ClassificationPreset()],
    include_tests=True,
)

classification_result = classification_report.run(
    reference_data=reference_data,
    current_data=current_data,
)

classification_result.save_html(ROOT_DIR + r"\classification_report.html")

In [ ]:
# Create DataDrift Report
drift_report = Report(
    metrics=[DataDriftPreset()])

datadrift_result = drift_report.run(
    reference_data=reference_data,
    current_data=current_data,
)
datadrift_result.save_html(ROOT_DIR + r"\drift_report.html")